In [1]:
import os
os.chdir(os.path.dirname(os.getcwd()))

import random
import hashlib
import simplejson as json
from datetime import datetime
from dotenv import load_dotenv
from datasets import Dataset

In [2]:
load_dotenv()

True

In [3]:
repo_id = "kenpath/mh-vistaar-claude"

In [4]:
def md5_sha_from_str(val):
    """Generate md5 hash from string"""
    return hashlib.md5(val.encode("utf-8")).hexdigest()

def md5_sha_from_dict(obj, ignore_nan=False, default=None,):
    """Generate md5 hash from dictionary"""
    json_data = json.dumps(obj, sort_keys=True,
                           ignore_nan=ignore_nan, default=default)
    return md5_sha_from_str(json_data)

    

def get_messages_from_record(record):
    # 1. Construct User Message
    user_str = f'**User:** "{record["question"]}"'
    agristack_str = "**Agristack Information Availability:** ✅" if record.get('agristack_required') == 'Yes' else "**Agristack Information Availability**: ❌"
    lang_str = "**Selected Language:** Marathi" # Assuming static based on prompt, or derive from record if available
    moderation_str = '**Moderation Recommendation:** Proceed with the query (Valid Agricultural)'
    
    user_msg = {
        "role": "user",
        "content": '\n'.join([user_str, lang_str, moderation_str, agristack_str])
    }

    # Initialize messages list
    messages = [user_msg]

    # 2. Process Agent Turns (The "In-between" logic)
    for turn in record.get('agent_turns', []):
        
        # Case A: The Assistant is speaking (Deciding to call tools)
        if turn['kind'] == 'response':
            current_tool_calls = []
            
            for part in turn['parts']:
                if part.get('part_kind') == 'tool-call':
                    # Create the tool call object matching your target format
                    tool_call_obj = {
                        "id": part['tool_call_id'],
                        "type": "function",
                        "function": {
                            "name": part['tool_name'],
                            # Target format requires arguments to be a JSON string, not a dict
                            "arguments": json.dumps(part['args']) 
                        }
                    }
                    current_tool_calls.append(tool_call_obj)
            
            # Only append the assistant message if there were actual tool calls
            if current_tool_calls:
                messages.append({
                    "role": "assistant",
                    "tool_calls": current_tool_calls
                })

        # Case B: The Tools are responding (Tool Outputs)
        elif turn['kind'] == 'request':
            for part in turn['parts']:
                if part.get('part_kind') == 'tool-return':
                    # Create a message for the tool result
                    # Note: Your target format uses "id" for the tool_call_id in the tool message
                    tool_msg = {
                        "role": "tool",
                        "content": part['content'],
                        "id": part['tool_call_id'],
                        "name": part['tool_name']
                    }
                    messages.append(tool_msg)

    # 3. Construct Final Assistant Answer
    final_msg = {
        "role": "assistant",
        "content": record['answer']
    }
    messages.append(final_msg)

    return messages


def get_unique_tool_names(record):
    unique_tools = set()
    
    # Iterate through every turn in the conversation history
    for turn in record.get('agent_turns', []):
        # Iterate through every part of the message (tool calls or tool returns)
        for part in turn.get('parts', []):
            # Safely get the tool name (in case there are pure text parts later)
            tool_name = part.get('tool_name')
            
            if tool_name:
                unique_tools.add(tool_name)

    unique_tools = list(unique_tools)
    unique_tools.sort()
    return unique_tools

### Run it to create and save dataset

In [5]:
models_dir = "data/models"

models_to_keep = [
    'claude-sonnet-4-5',
    'claude-opus-4-5',
                 ]

In [9]:
hf_records = []
for model in models_to_keep:
    data     = json.load(open(os.path.join(models_dir,model,"data.json")))
    metadata = json.load(open(os.path.join(models_dir,model,"metadata.json")))
    ts        = datetime.fromisoformat(metadata['date'])
    for record in data:
        msgs      = get_messages_from_record(record)
        id_suffix = md5_sha_from_dict(msgs)
        hf_record = {
            "id":model+id_suffix,
            "timestamp":ts,
            "messages_json":json.dumps(msgs,ensure_ascii=False),
            "tools":get_unique_tool_names(record),
            "model":model            
        }
        hf_records.append(hf_record)
        
dataset = Dataset.from_list(hf_records)
print(f"Preparing to push {len(hf_records)} records to {repo_id}...")

Preparing to push 940 records to kenpath/mh-vistaar-claude...


In [10]:

try:
    dataset.push_to_hub(repo_id, 
                        private=True,
                        token=os.getenv("HF_TOKEN")
                       )
    print(f"✅ Successfully pushed to https://huggingface.co/datasets/{repo_id}")
except Exception as e:
    print(f"❌ Error pushing to Hub: {e}")
    print("Tip: Make sure you are logged in. Run `huggingface-cli login` in terminal")
    print("or use `from huggingface_hub import login; login('YOUR_WRITE_TOKEN')` in the script.")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/427 [00:00<?, ?B/s]

✅ Successfully pushed to https://huggingface.co/datasets/kenpath/mh-vistaar-claude
